### Cleaning

In [2]:
import pandas as pd
import numpy as np

In [3]:
funds = {
    "Nippon India Large Cap": "Large_cap/nippon.xlsx",
    "ICICI Prudential Large Cap": "Large_cap/ICICI.xlsx",
    "HDFC Large Cap": "Large_cap/hdfc.xlsx",
    "DSP Large Cap": "Large_cap/dsp.xlsx",
    "Bandhan Large Cap": "Large_cap/bandhan.xlsx"
}

In [4]:
def clean_nav_data(file_path):

    # --------------------------------------------------
    # 1. Read Excel without assuming header position
    # --------------------------------------------------
    raw = pd.read_excel(file_path, header=None)

    # --------------------------------------------------
    # 2. Find the row containing "Net Asset Value"
    # --------------------------------------------------
    header_row = None

    for i in range(len(raw)):
        row = raw.iloc[i].astype(str).str.strip().tolist()

        if "Net Asset Value" in row:
            header_row = i
            break

    if header_row is None:
        raise ValueError(
            f"Could not find the header row in {file_path}"
        )

    print(f"{file_path} -> header row: {header_row}")

    # --------------------------------------------------
    # 3. Read Excel again using the detected header
    # --------------------------------------------------
    df = pd.read_excel(
        file_path,
        header=header_row
    )

    # Clean column names
    df.columns = (
        df.columns
        .astype(str)
        .str.strip()
    )

    # --------------------------------------------------
    # 4. Keep required columns
    # --------------------------------------------------
    df = df[
        [
            "Date",
            "Net Asset Value",
            "Plan",
            "Option"
        ]
    ]

    # Rename NAV
    df = df.rename(
        columns={
            "Net Asset Value": "NAV"
        }
    )

    # --------------------------------------------------
    # 5. Clean data types
    # --------------------------------------------------
    df["Date"] = pd.to_datetime(
        df["Date"],
        errors="coerce"
    )

    df["NAV"] = pd.to_numeric(
        df["NAV"],
        errors="coerce"
    )

    df["Plan"] = (
        df["Plan"]
        .astype(str)
        .str.strip()
    )

    df["Option"] = (
        df["Option"]
        .astype(str)
        .str.strip()
    )

    # --------------------------------------------------
    # 6. Keep Direct Plan + Growth Option
    # --------------------------------------------------
    df = df[
        df["Plan"].str.contains("direct", case=False, na=False) &
        df["Option"].str.contains("growth", case=False, na=False)
    ]

    # --------------------------------------------------
    # 7. Keep 2022–2025
    # --------------------------------------------------
    df = df[
        (df["Date"] >= "2022-01-01") &
        (df["Date"] <= "2025-12-31")
    ]

    # --------------------------------------------------
    # 8. Remove missing Date/NAV
    # --------------------------------------------------
    df = df.dropna(
        subset=["Date", "NAV"]
    )

    # --------------------------------------------------
    # 9. Sort chronologically
    # --------------------------------------------------
    df = (
        df
        .sort_values("Date")
        .reset_index(drop=True)
    )

    return df

In [5]:
cleaned_data = {}

for fund_name, file_path in funds.items():

    cleaned_data[fund_name] = clean_nav_data(file_path)

print("Cleaning completed.")

Large_cap/nippon.xlsx -> header row: 4
Large_cap/ICICI.xlsx -> header row: 4
Large_cap/hdfc.xlsx -> header row: 4
Large_cap/dsp.xlsx -> header row: 4
Large_cap/bandhan.xlsx -> header row: 4
Cleaning completed.


In [6]:
cleaned_data["Nippon India Large Cap"]

,Date,NAV,Plan,Option
0,2022-01-03,54.3991,Direct Plan,Growth Option
1,2022-01-04,54.9154,Direct Plan,Growth Option
2,2022-01-05,55.1480,Direct Plan,Growth Option
3,2022-01-06,54.8331,Direct Plan,Growth Option
4,2022-01-07,54.8875,Direct Plan,Growth Option
...,...,...,...,...
980,2025-12-24,105.3548,Direct Plan,Growth Option
981,2025-12-26,104.9912,Direct Plan,Growth Option
982,2025-12-29,104.7188,Direct Plan,Growth Option
983,2025-12-30,104.5935,Direct Plan,Growth Option


In [7]:
for fund_name, df in cleaned_data.items():

    print("\n" + "=" * 60)
    print(fund_name)
    print("=" * 60)

    print("Rows:", len(df))
    print("Start date:", df["Date"].min())
    print("End date:", df["Date"].max())

    print("Missing dates:", df["Date"].isna().sum())
    print("Missing NAV:", df["NAV"].isna().sum())

    print("Duplicate dates:", df["Date"].duplicated().sum())

    print("Plans:", df["Plan"].unique())
    print("Options:", df["Option"].unique())


Nippon India Large Cap
Rows: 985
Start date: 2022-01-03 00:00:00
End date: 2025-12-31 00:00:00
Missing dates: 0
Missing NAV: 0
Duplicate dates: 0
Plans: ['Direct Plan']
Options: ['Growth Option']

ICICI Prudential Large Cap
Rows: 985
Start date: 2022-01-03 00:00:00
End date: 2025-12-31 00:00:00
Missing dates: 0
Missing NAV: 0
Duplicate dates: 0
Plans: ['Direct Plan']
Options: ['Growth']

HDFC Large Cap
Rows: 985
Start date: 2022-01-03 00:00:00
End date: 2025-12-31 00:00:00
Missing dates: 0
Missing NAV: 0
Duplicate dates: 0
Plans: ['Direct Plan']
Options: ['Growth Option']

DSP Large Cap
Rows: 985
Start date: 2022-01-03 00:00:00
End date: 2025-12-31 00:00:00
Missing dates: 0
Missing NAV: 0
Duplicate dates: 0
Plans: ['Direct Plan']
Options: ['Growth']

Bandhan Large Cap
Rows: 991
Start date: 2022-01-03 00:00:00
End date: 2025-12-31 00:00:00
Missing dates: 0
Missing NAV: 0
Duplicate dates: 0
Plans: ['Direct Plan']
Options: ['Growth']


## Saving

In [8]:
for fund_name in cleaned_data:

    cleaned_data[fund_name] = (
        cleaned_data[fund_name]
        .reset_index(drop=True)
    )

In [9]:
import os

os.makedirs("Large_cap/clean", exist_ok=True)

In [10]:
for fund_name, df in cleaned_data.items():

    filename = (
        fund_name
        .lower()
        .replace(" ", "_")
        .replace("india_", "")
        .replace("prudential_", "")
    )

    output_path = f"Large_cap/clean/{filename}.csv"

    df.to_csv(output_path, index=False)

    print(f"Saved: {output_path}")

Saved: Large_cap/clean/nippon_large_cap.csv
Saved: Large_cap/clean/icici_large_cap.csv
Saved: Large_cap/clean/hdfc_large_cap.csv
Saved: Large_cap/clean/dsp_large_cap.csv
Saved: Large_cap/clean/bandhan_large_cap.csv


### Create daily returns

In [11]:
cleaned_files = {
    "Nippon India Large Cap": "Large_cap/clean/nippon_large_cap.csv",
    "ICICI Prudential Large Cap": "Large_cap/clean/icici_large_cap.csv",
    "HDFC Large Cap": "Large_cap/clean/hdfc_large_cap.csv",
    "DSP Large Cap": "Large_cap/clean/dsp_large_cap.csv",
    "Bandhan Large Cap": "Large_cap/clean/bandhan_large_cap.csv"
}

In [12]:
analysis_data = {}

for fund_name, file_path in cleaned_files.items():

    df = pd.read_csv(file_path)

    df["Date"] = pd.to_datetime(df["Date"])

    # Daily return
    df["Daily_Return"] = df["NAV"].pct_change()

    analysis_data[fund_name] = df

In [13]:
display(analysis_data["Nippon India Large Cap"].head())

,Date,NAV,Plan,Option,Daily_Return
0,2022-01-03,54.3991,Direct Plan,Growth Option,NaN
1,2022-01-04,54.9154,Direct Plan,Growth Option,0.009491
2,2022-01-05,55.1480,Direct Plan,Growth Option,0.004236
3,2022-01-06,54.8331,Direct Plan,Growth Option,-0.005710
4,2022-01-07,54.8875,Direct Plan,Growth Option,0.000992


### Read them and create the analysis columns

In [14]:
analysis_data = {}

for fund_name, file_path in cleaned_files.items():

    df = pd.read_csv(file_path)

    df["Date"] = pd.to_datetime(df["Date"])

    # Daily return
    df["Daily_Return"] = df["NAV"].pct_change()

    # Running maximum NAV
    df["Running_Max"] = df["NAV"].cummax()

    # Drawdown
    df["Drawdown"] = (
        (df["NAV"] - df["Running_Max"])
        / df["Running_Max"]
    )

    analysis_data[fund_name] = df

In [15]:
display(analysis_data["Nippon India Large Cap"].head())

,Date,NAV,Plan,Option,Daily_Return,Running_Max,Drawdown
0,2022-01-03,54.3991,Direct Plan,Growth Option,NaN,54.3991,0.000000
1,2022-01-04,54.9154,Direct Plan,Growth Option,0.009491,54.9154,0.000000
2,2022-01-05,55.1480,Direct Plan,Growth Option,0.004236,55.1480,0.000000
3,2022-01-06,54.8331,Direct Plan,Growth Option,-0.005710,55.1480,-0.005710
4,2022-01-07,54.8875,Direct Plan,Growth Option,0.000992,55.1480,-0.004724


### Save these enhanced datasets too

In [16]:
import os

os.makedirs("Large_cap/analysis", exist_ok=True)

for fund_name, df in analysis_data.items():

    filename = (
        fund_name
        .lower()
        .replace(" ", "_")
        .replace("india_", "")
        .replace("prudential_", "")
    )

    output_path = f"Large_cap/analysis/{filename}_analysis.csv"

    df.to_csv(output_path, index=False)

    print("Saved:", output_path)

Saved: Large_cap/analysis/nippon_large_cap_analysis.csv
Saved: Large_cap/analysis/icici_large_cap_analysis.csv
Saved: Large_cap/analysis/hdfc_large_cap_analysis.csv
Saved: Large_cap/analysis/dsp_large_cap_analysis.csv
Saved: Large_cap/analysis/bandhan_large_cap_analysis.csv


### Analysis

In [17]:
analysis_files = {
    "Nippon India Large Cap": "Large_cap/analysis/nippon_large_cap_analysis.csv",
    "ICICI Prudential Large Cap": "Large_cap/analysis/icici_large_cap_analysis.csv",
    "HDFC Large Cap": "Large_cap/analysis/hdfc_large_cap_analysis.csv",
    "DSP Large Cap": "Large_cap/analysis/dsp_large_cap_analysis.csv",
    "Bandhan Large Cap": "Large_cap/analysis/bandhan_large_cap_analysis.csv"
}

In [18]:
for fund_name, file_path in analysis_files.items():

    df = pd.read_csv(file_path)
    df["Date"] = pd.to_datetime(df["Date"])

    print(
        f"{fund_name}: "
        f"{df['Date'].min().date()} → {df['Date'].max().date()} "
        f"({len(df)} observations)"
    )

Nippon India Large Cap: 2022-01-03 → 2025-12-31 (985 observations)
ICICI Prudential Large Cap: 2022-01-03 → 2025-12-31 (985 observations)
HDFC Large Cap: 2022-01-03 → 2025-12-31 (985 observations)
DSP Large Cap: 2022-01-03 → 2025-12-31 (985 observations)
Bandhan Large Cap: 2022-01-03 → 2025-12-31 (991 observations)


In [19]:
start_dates = []
end_dates = []

for file_path in analysis_files.values():

    df = pd.read_csv(file_path)
    df["Date"] = pd.to_datetime(df["Date"])

    start_dates.append(df["Date"].min())
    end_dates.append(df["Date"].max())

common_start = max(start_dates)
common_end = min(end_dates)

print("Common period:")
print(common_start.date(), "→", common_end.date())

Common period:
2022-01-03 → 2025-12-31


### Calculate CAGR

In [20]:
def calculate_cagr(df):

    start_nav = df["NAV"].iloc[0]
    end_nav = df["NAV"].iloc[-1]

    years = (
        df["Date"].iloc[-1] -
        df["Date"].iloc[0]
    ).days / 365.25

    return (
        (end_nav / start_nav) ** (1 / years)
    ) - 1

### Calculate annualized volatility

In [21]:
def calculate_volatility(df):

    return (
        df["Daily_Return"]
        .dropna()
        .std()
        * np.sqrt(252)
    )

### Calculate maximum drawdown

In [22]:
def calculate_max_drawdown(df):

    return df["Drawdown"].min()

### Create the Large Cap fund-level results

In [23]:
results = []

for fund_name, df in analysis_data.items():

    results.append({
        "Fund": fund_name,
        "Start Date": df["Date"].min(),
        "End Date": df["Date"].max(),
        "Observations": len(df),
        "Start NAV": df["NAV"].iloc[0],
        "End NAV": df["NAV"].iloc[-1],
        "CAGR": calculate_cagr(df),
        "Volatility": calculate_volatility(df),
        "Maximum Drawdown": calculate_max_drawdown(df)
    })

large_cap_results = pd.DataFrame(results)

display(large_cap_results)

,Fund,Start Date,End Date,Observations,Start NAV,End NAV,CAGR,Volatility,Maximum Drawdown
0,Nippon India Large Cap,2022-01-03,2025-12-31,985,54.3991,105.3421,0.180049,0.136721,-0.153680
1,ICICI Prudential Large Cap,2022-01-03,2025-12-31,985,71.0800,127.1700,0.156883,0.123998,-0.153933
2,HDFC Large Cap,2022-01-03,2025-12-31,985,732.7530,1279.9530,0.149963,0.131588,-0.163646
3,DSP Large Cap,2022-01-03,2025-12-31,985,312.6130,534.9380,0.144048,0.124441,-0.176848
4,Bandhan Large Cap,2022-01-03,2025-12-31,991,56.4500,92.7700,0.132523,0.135343,-0.194181


### Save the updated analysis files

In [24]:
os.makedirs("Large_cap/analysis", exist_ok=True)

for fund_name, df in analysis_data.items():

    filename = (
        fund_name
        .lower()
        .replace(" ", "_")
        .replace("india_", "")
        .replace("prudential_", "")
    )

    path = f"Large_cap/analysis/{filename}_analysis.csv"

    df.to_csv(path, index=False)

    print("Saved:", path)

Saved: Large_cap/analysis/nippon_large_cap_analysis.csv
Saved: Large_cap/analysis/icici_large_cap_analysis.csv
Saved: Large_cap/analysis/hdfc_large_cap_analysis.csv
Saved: Large_cap/analysis/dsp_large_cap_analysis.csv
Saved: Large_cap/analysis/bandhan_large_cap_analysis.csv


In [25]:
large_cap_results.to_csv(
    "Large_cap/analysis/large_cap_fund_results.csv",
    index=False
)

### Sharpe Ratio

In [29]:

import numpy as np
import pandas as pd

# Assumptions
RISK_FREE_RATE = 0.06
TRADING_DAYS = 252

# Calculate the conventional annualized Sharpe ratio
def calculate_sharpe(df, risk_free_rate=RISK_FREE_RATE):
    daily_returns = df["Daily_Return"].dropna()

    if len(daily_returns) < 2:
        return np.nan

    daily_rf = (1 + risk_free_rate) ** (1 / TRADING_DAYS) - 1

    excess_returns = daily_returns - daily_rf
    daily_std = daily_returns.std()

    if pd.isna(daily_std) or daily_std == 0:
        return np.nan

    return (excess_returns.mean() / daily_std) * np.sqrt(TRADING_DAYS)


# Add Sharpe Ratio to the existing results DataFrame
large_cap_results["Sharpe Ratio"] = [
    calculate_sharpe(common_data[fund_name])
    for fund_name in large_cap_results["Fund"]
]

# Save updated results to the existing CSV file
output_path = "Large_cap/analysis/large_cap_fund_results.csv"

large_cap_results.to_csv(output_path, index=False)

# Display final results
display(large_cap_results)

print(f"Updated results saved to: {output_path}")


,Fund,Start Date,End Date,Observations,Start NAV,End NAV,CAGR,Volatility,Maximum Drawdown,Sharpe Ratio
0,Nippon India Large Cap,2022-01-03,2025-12-31,985,54.3991,105.3421,0.180049,0.136721,-0.153680,0.880623
1,ICICI Prudential Large Cap,2022-01-03,2025-12-31,985,71.0800,127.1700,0.156883,0.123998,-0.153933,0.793930
2,HDFC Large Cap,2022-01-03,2025-12-31,985,732.7530,1279.9530,0.149963,0.131588,-0.163646,0.708972
3,DSP Large Cap,2022-01-03,2025-12-31,985,312.6130,534.9380,0.144048,0.124441,-0.176848,0.699834
4,Bandhan Large Cap,2022-01-03,2025-12-31,991,56.4500,92.7700,0.132523,0.135343,-0.194181,0.571672


Updated results saved to: Large_cap/analysis/large_cap_fund_results.csv
